# Lab Assignment 4 – NLP Preprocessing and Text Classification

**Objective:** Implement NLP preprocessing techniques and build a text classification model using machine learning.

## Learning Outcomes
- Apply NLP preprocessing (tokenization, stopword removal, stemming, lemmatization)
- Implement text vectorization (TF-IDF, CountVectorizer)
- Build a machine learning classification model
- Evaluate model performance using metrics

## Section 1 – Imports & Setup

In [ ]:
import re, string, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for pkg in ['punkt','punkt_tab','stopwords','wordnet','averaged_perceptron_tagger','omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

print('All libraries loaded successfully!')

## Section 2 – Load Dataset

In [ ]:
categories = ['rec.sport.hockey', 'sci.med', 'comp.graphics', 'talk.politics.misc']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers','footers','quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers','footers','quotes'))

print(f'Training samples : {len(train_data.data)}')
print(f'Testing  samples : {len(test_data.data)}')
print(f'Categories       : {train_data.target_names}')
print(f'\nSample text:\n{train_data.data[0][:400]}')

## Section 3 – NLP Preprocessing

### 3a Tokenization

In [ ]:
sample = 'The doctors are studying 3 new medical treatments for COVID-19 patients!'
tokens = word_tokenize(sample.lower())
print('Tokens:', tokens)

### 3b Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))
filtered = [w for w in tokens if w not in stop_words and w.isalpha()]
print('After stopword removal:', filtered)

### 3c Stemming

In [ ]:
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in filtered]
print('Stemmed:', stemmed)

### 3d Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in filtered]
print('Lemmatized:', lemmatized)

### 3e Full Preprocessing Pipeline

In [ ]:
def preprocess_text(text, use_stemming=False, use_lemmatization=True):
    """Complete NLP preprocessing pipeline."""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'\S+@\S+', '', text)                  # remove emails
    text = re.sub(r'\d+', '', text)                       # remove numbers
    text = text.translate(str.maketrans('','',string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    elif use_lemmatization:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

print('Original :', sample)
print('Cleaned  :', preprocess_text(sample))

In [ ]:
# Preprocess all data
X_train_clean = [preprocess_text(t) for t in train_data.data]
X_test_clean  = [preprocess_text(t) for t in test_data.data]
y_train, y_test = train_data.target, test_data.target
print('Preprocessing complete.')
print('Sample cleaned text:', X_train_clean[0][:200])

## Section 4 – Text Vectorization

### 4a CountVectorizer

In [ ]:
count_vec = CountVectorizer(max_features=10000, ngram_range=(1,2))
X_train_count = count_vec.fit_transform(X_train_clean)
X_test_count  = count_vec.transform(X_test_clean)
print(f'CountVectorizer matrix : {X_train_count.shape}')
print(f'Sample features        : {count_vec.get_feature_names_out()[:10].tolist()}')

### 4b TF-IDF Vectorizer

In [ ]:
tfidf_vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_train_tfidf = tfidf_vec.fit_transform(X_train_clean)
X_test_tfidf  = tfidf_vec.transform(X_test_clean)
print(f'TF-IDF matrix : {X_train_tfidf.shape}')

print('\nTop 10 TF-IDF terms per category:')
for i, cat in enumerate(train_data.target_names):
    idx   = np.where(y_train == i)[0]
    mean  = np.asarray(X_train_tfidf[idx].mean(axis=0)).flatten()
    top10 = np.argsort(mean)[-10:][::-1]
    terms = tfidf_vec.get_feature_names_out()[top10]
    print(f'  {cat:30s}: {", ".join(terms)}')

## Section 5 – Model Training & Evaluation

In [ ]:
models = {
    'Naive Bayes (Count)':  (MultinomialNB(),                          X_train_count, X_test_count),
    'Naive Bayes (TF-IDF)': (MultinomialNB(),                          X_train_tfidf, X_test_tfidf),
    'Logistic Regression':  (LogisticRegression(max_iter=1000, C=1.0), X_train_tfidf, X_test_tfidf),
    'Linear SVM':           (LinearSVC(max_iter=2000,  C=1.0),         X_train_tfidf, X_test_tfidf),
}

results = {}
for name, (clf, Xtr, Xte) in models.items():
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xte)
    acc, f1 = accuracy_score(y_test, y_pred), f1_score(y_test, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1': f1, 'clf': clf, 'y_pred': y_pred}
    print(f'{name:30s} | Accuracy: {acc:.4f} | F1: {f1:.4f}')

## Section 6 – Detailed Evaluation (Best Model)

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best      = results[best_name]
print(f'Best model: {best_name}\n')
print(classification_report(y_test, best['y_pred'], target_names=train_data.target_names))

## Section 7 – Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('NLP Text Classification – Results', fontsize=16, fontweight='bold')

# Model accuracy comparison
ax = axes[0,0]
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
bars  = ax.barh(names, accs, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_xlim(0, 1.05); ax.set_xlabel('Accuracy'); ax.set_title('Model Accuracy Comparison')
for bar, acc in zip(bars, accs):
    ax.text(acc+0.005, bar.get_y()+bar.get_height()/2, f'{acc:.3f}', va='center', fontsize=10)

# Confusion matrix
ax = axes[0,1]
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[c.split('.')[-1] for c in train_data.target_names],
            yticklabels=[c.split('.')[-1] for c in train_data.target_names])
ax.set_title(f'Confusion Matrix – {best_name}'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# Class distribution
ax = axes[1,0]
counts = pd.Series(y_train).value_counts().sort_index()
ax.bar([train_data.target_names[i].split('.')[-1] for i in counts.index],
       counts.values, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_title('Training Set Class Distribution'); ax.set_xlabel('Category'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)

# Per-category F1
ax = axes[1,1]
report = classification_report(y_test, best['y_pred'],
                                target_names=train_data.target_names, output_dict=True)
cat_f1  = [report[c]['f1-score'] for c in train_data.target_names]
cat_lbl = [c.split('.')[-1] for c in train_data.target_names]
ax.bar(cat_lbl, cat_f1, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_ylim(0,1.05); ax.set_title(f'Per-Category F1 – {best_name}')
ax.set_xlabel('Category'); ax.set_ylabel('F1 Score'); ax.tick_params(axis='x', rotation=15)
for i,v in enumerate(cat_f1): ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 8 – Predict on Custom Text

In [ ]:
custom_texts = [
    'The hockey player scored a hat trick in the final game of the playoffs.',
    'The MRI scan revealed a tumor in the patient\'s brain requiring immediate surgery.',
    'OpenGL provides APIs for rendering 2D and 3D vector graphics on GPU hardware.',
    'The senator\'s speech on immigration policy sparked heated debate in Congress.',
]

for txt in custom_texts:
    cleaned  = preprocess_text(txt)
    vec      = tfidf_vec.transform([cleaned])
    pred_idx = best['clf'].predict(vec)[0]
    pred_cat = train_data.target_names[pred_idx]
    print(f'Text     : {txt[:70]}...')
    print(f'Predicted: {pred_cat}\n')

## Section 9 – Sklearn Pipeline

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)),
    ('clf',   LinearSVC(max_iter=2000, C=1.0)),
])
pipeline.fit(X_train_clean, y_train)
print(f'Pipeline accuracy: {accuracy_score(y_test, pipeline.predict(X_test_clean)):.4f}')